# Basic FastHTML Integration Example

> Demonstrates real-time tqdm progress tracking in FastHTML using HTMX polling. Shows how to capture tqdm output from background tasks, display live progress bars with percentage and status updates, and handle task completion with automatic UI state management.

In [1]:
from fasthtml.common import *
import uuid, time

from cjm_tqdm_capture.progress_monitor import ProgressMonitor
from cjm_tqdm_capture.job_runner import JobRunner

In [2]:
# For Jupyter display
from fasthtml.jupyter import JupyUvi, HTMX
from cjm_fasthtml_daisyui.core.testing import create_test_app, create_test_page, start_test_server
from cjm_fasthtml_daisyui.core.themes import DaisyUITheme
from IPython.display import display

# Import DaisyUI factories
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_colors, btn_behaviors
from cjm_fasthtml_daisyui.components.feedback.progress import progress, progress_colors

# Import Tailwind factories
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.typography import font_weight, font_size
from cjm_fasthtml_tailwind.utilities.sizing import w
from cjm_fasthtml_tailwind.core.base import combine_classes

In [3]:
# Create app
app, rt = create_test_app(theme=DaisyUITheme.LIGHT)

# Initialize monitor and runner
monitor = ProgressMonitor()
runner = JobRunner(monitor)

In [4]:
# Define a simple task that uses tqdm
def simple_task():
    from tqdm import tqdm
    import time
    
    results = []
    for i in tqdm(range(100), desc="Processing"):
        time.sleep(0.01)
        results.append(i * 2)
    return results

In [5]:
# Create helper function for Progress bars
def create_progress_bar(value="0", max="100"):
    """Create a styled progress bar with consistent styling"""
    return Progress(
        value=str(value), 
        max=str(max), 
        cls=combine_classes(progress, progress_colors.primary, w.full)
    )

In [6]:
# Create helper function for Progress labels
def create_progress_label(text="Progress:"):
    """Create a styled progress label with consistent styling"""
    return P(text, cls=combine_classes(font_weight.bold))

In [7]:
# Create helper function for status messages
def create_status_message(text):
    """Create a styled status message with consistent styling"""
    return P(text, cls=combine_classes(m.t(2), font_size.sm))

In [8]:
# Create helper function for complete progress display
def create_progress_display(value="0", status_text="Waiting...", **div_attrs):
    """Create a complete progress display with label, bar, and status"""
    return Div(
        create_progress_label(),
        create_progress_bar(value=value, max="100"),
        create_status_message(status_text),
        **div_attrs
    )

In [9]:
# Helper function to add HTMX polling attributes
def with_polling(job_id, delay="100ms"):
    """Return HTMX polling attributes for job progress"""
    return {
        "hx_get": f"/job_progress?job_id={job_id}",
        "hx_trigger": f"load delay:{delay}",
        "hx_swap": "outerHTML"
    }

In [10]:
# Helper to extract progress info from snapshot
def get_progress_info(snapshot):
    """Extract progress information from a snapshot"""
    if not snapshot:
        return 0, "Waiting..."
    
    if snapshot['completed']:
        return 100, "Completed!"
    
    # Get first progress bar info if available
    if snapshot['bars']:
        first_bar = next(iter(snapshot['bars'].values()))
        status = f"{first_bar.description}: {first_bar.progress:.1f}% ({first_bar.current}/{first_bar.total})"
    else:
        status = f"Processing... {snapshot['overall_progress']:.1f}%"
    
    return snapshot['overall_progress'], status

In [11]:
# Create minimal UI with HTMX polling
def render_start_button(disabled=False):
    """Render start button with appropriate state"""
    btn_classes = combine_classes(
        btn, 
        btn_colors.primary,
        btn_behaviors.disabled if disabled else ""
    )
    
    return Button(
        "Start Task", 
        id="start-btn",
        hx_post="/start_task" if not disabled else None,
        hx_target="#progress-container",
        hx_swap="innerHTML",
        cls=btn_classes,
        disabled=disabled,
        hx_swap_oob="true"  # Enable out-of-band swap
    )

In [12]:
@rt("/")
def index():
    return create_test_page(
        "Basic Progress Demo",
        Div(
            H2("Simple Progress Tracking"),
            render_start_button(disabled=False),
            Div(
                P("Ready", cls=combine_classes(font_weight.bold, m.t(4))),
                id="progress-container",
                cls=str(m.t(4))
            ),
            cls=str(p(8))
        )
    )

In [13]:
# API endpoints using HTMX polling with conditional updates
@rt("/start_task")
def start_task():
    job_id = str(uuid.uuid4())
    
    runner.start(
        job_id, 
        simple_task,
        patch_kwargs={
            "min_delta_pct": 5,
            "min_update_interval": 0.05,
            "emit_initial": True
        }
    )
    
    # Return initial UI with disabled button via out-of-band swap
    return Div(
        # Disable button via out-of-band swap
        render_start_button(disabled=True),
        # Progress display with initial load trigger
        create_progress_display(
            value="0",
            status_text="Starting...",
            hx_get=f"/job_progress?job_id={job_id}",
            hx_trigger="load",
            hx_swap="innerHTML"
        )
    )

In [14]:
@rt("/job_progress")
def job_progress(job_id: str):
    """Returns current progress for a job"""
    snapshot = monitor.snapshot(job_id)
    progress_value, status_text = get_progress_info(snapshot)
    
    # Handle completed state
    if snapshot and snapshot['completed']:
        return Div(
            # Re-enable button via out-of-band swap
            render_start_button(disabled=False),
            # Final progress display without polling
            create_progress_display(value="100", status_text="Completed!")
        )
    
    # Continue polling for active or pending jobs
    return create_progress_display(
        value=str(progress_value),
        status_text=status_text,
        **with_polling(job_id)
    )

In [15]:
# Start server for Jupyter
server = start_test_server(app)
display(HTMX(port=server.port))

In [16]:
# Stop server when done
server.stop()